# Train the Card Model: segmentation (run in Google Colab, ~15 min)

Same data you already collected (`card_dataset.zip`), new kind of model: **YOLO26n-seg** finds each card's
**outline**, not just a box. That keeps overlapping, turned cards apart (boxes around tilted overlapping cards
cover each other, which is what broke the readings).

**Before you start:** *Runtime > Change runtime type > T4 GPU*. Then run everything top to bottom.

Steps:
1. Upload `card_dataset.zip`
2. **Turn the boxes into card outlines automatically** (each picture has one card on the table)
3. Split train / val (same held-out pictures as before)
4. **Make synthetic dealt hands**: real cards cut out, turned, and laid overlapping, like a dealer deals
5. Train for `TRAIN_MINUTES` (15) with copy-paste + rotation augmentation
6. Check accuracy on single cards **and on dealt hands**
7. Export `best.onnx` (opset 12) and download it -> put it in `HackRPI26/models/` on the Jetson

The Jetson code (`card_reader`, `blackjack_game`) already reads this new model.

In [ ]:
!pip install -q -U ultralytics

import os, glob, random, shutil, zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

CLASSES = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'DOWN']   # same order as the collector
IMGSZ = 416                     # what the Jetson runs
START_WEIGHTS = 'yolo26n-seg.pt'   # newest YOLO, nano, SEGMENTATION. Use 'yolo11n-seg.pt' if this fails to download
TRAIN_MINUTES = 15
os.chdir('/content')

## 1. Upload our dataset
On the Jetson, zip the folder first (in a Jupyter terminal, inside `HackRPI26/`):
```
zip -r card_dataset.zip card_dataset
```
Download that zip to your laptop, then upload it with the cell below.

In [ ]:
from google.colab import files
uploaded = files.upload()   # pick card_dataset.zip

shutil.rmtree('/content/card_dataset', ignore_errors=True)
with zipfile.ZipFile('card_dataset.zip') as z:
    z.extractall('/content')
print("Images:", len(glob.glob('/content/card_dataset/images/*.jpg')))

## 2. Boxes -> card outlines (automatic)
Inside each labeled box, everything that doesn't look like the table is the card. That gets fit to 4 corners.
If it can't find a clean card shape, it keeps the box (counted below).

In [ ]:
def read_box_label(path, w, h):
    c, xc, yc, bw, bh = open(path).read().split()[:5]
    xc, yc, bw, bh = float(xc) * w, float(yc) * h, float(bw) * w, float(bh) * h
    return int(c), (xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2)


def card_polygon(img, box, pad=0.12):
    # The card's outline (4 corners, pixels) inside its box. Card = pixels that don't look like the table.
    # The table color is taken from the edge of a slightly bigger crop around the box.
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    px, py = (x2 - x1) * pad, (y2 - y1) * pad
    X1, Y1 = int(max(0, x1 - px)), int(max(0, y1 - py))
    X2, Y2 = int(min(w, x2 + px)), int(min(h, y2 + py))
    crop = cv2.GaussianBlur(img[Y1:Y2, X1:X2], (5, 5), 0)
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(np.float32)
    edge = np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])
    table = np.median(edge, axis=0)
    mask = (np.linalg.norm(lab - table, axis=2) > 30).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    box_area = (x2 - x1) * (y2 - y1)
    if contours:
        hull = cv2.convexHull(max(contours, key=cv2.contourArea))
        quad = None
        for eps in (0.02, 0.03, 0.05, 0.08):
            approx = cv2.approxPolyDP(hull, eps * cv2.arcLength(hull, True), True)
            if len(approx) == 4:
                quad = approx.reshape(4, 2).astype(np.float32)
                break
        if quad is None:
            quad = cv2.boxPoints(cv2.minAreaRect(hull)).astype(np.float32)
        quad += [X1, Y1]
        if 0.4 * box_area <= cv2.contourArea(quad) <= 1.3 * box_area:
            return quad, True
    # Fallback: the box itself
    return np.array([(x1, y1), (x2, y1), (x2, y2), (x1, y2)], np.float32), False


def write_seg_label(path, items, w, h):
    # items = [(class, polygon pixels)] -> YOLO segmentation format: class x1 y1 x2 y2 ... (0-1)
    with open(path, 'w') as f:
        for c, poly in items:
            pts = ' '.join(f"{x / w:.6f} {y / h:.6f}" for x, y in np.clip(poly, 0, [w - 1, h - 1]))
            f.write(f"{c} {pts}\n")


SRC = '/content/card_dataset'
ALL = '/content/seg_all'
shutil.rmtree(ALL, ignore_errors=True)
os.makedirs(f'{ALL}/images'); os.makedirs(f'{ALL}/labels')
fell_back = []
for img_path in sorted(glob.glob(f'{SRC}/images/*.jpg')):
    name = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    c, box = read_box_label(f'{SRC}/labels/{name}.txt', w, h)
    poly, ok = card_polygon(img, box)
    if not ok:
        fell_back.append(name)
    shutil.copy(img_path, f'{ALL}/images/')
    write_seg_label(f'{ALL}/labels/{name}.txt', [(c, poly)], w, h)
n = len(glob.glob(f'{ALL}/images/*.jpg'))
print(f"Converted {n} images. Card outline found automatically in {n - len(fell_back)}; used the box for {len(fell_back)}.")

Check some outlines (green). A few `BOX FALLBACK` ones are fine; lots of them means the table color is fooling it.

In [ ]:
def read_seg_label(path, w, h):
    items = []
    for line in open(path).read().splitlines():
        v = line.split()
        items.append((int(v[0]), (np.array(v[1:], float).reshape(-1, 2) * [w, h]).astype(np.float32)))
    return items


def draw_items(img, items, title):
    img = img.copy()
    for c, poly in items:
        cv2.polylines(img, [poly.astype(np.int32)], True, (0, 255, 0), 2)
        x, y = poly.min(0).astype(int)
        cv2.putText(img, CLASSES[c], (x, max(y - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title(title, fontsize=8); plt.axis('off')


names = [os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{ALL}/images/*.jpg')]
show = random.Random(1).sample(names, min(8, len(names))) + fell_back[:4]
plt.figure(figsize=(18, 9))
for i, name in enumerate(show[:12]):
    img = cv2.imread(f'{ALL}/images/{name}.jpg')
    plt.subplot(3, 4, i + 1)
    draw_items(img, read_seg_label(f'{ALL}/labels/{name}.txt', img.shape[1], img.shape[0]),
               name + ('  (BOX FALLBACK)' if name in fell_back else ''))
plt.tight_layout(); plt.show()

## 3. Split into train / val (80 / 20)

In [ ]:
def split_dataset(src, dst, val_frac=0.2, seed=0):
    names = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{src}/images/*.jpg'))
    random.Random(seed).shuffle(names)
    n_val = max(1, int(len(names) * val_frac))
    for split, group in [('val', names[:n_val]), ('train', names[n_val:])]:
        os.makedirs(f'{dst}/{split}/images', exist_ok=True)
        os.makedirs(f'{dst}/{split}/labels', exist_ok=True)
        for n in group:
            shutil.copy(f'{src}/images/{n}.jpg', f'{dst}/{split}/images/')
            shutil.copy(f'{src}/labels/{n}.txt', f'{dst}/{split}/labels/')
    return len(names) - n_val, n_val


shutil.rmtree('/content/seg', ignore_errors=True)
n_train, n_val = split_dataset(ALL, '/content/seg')   # same seed as before: same held-out pictures
with open('/content/seg.yaml', 'w') as f:
    f.write(f"path: /content/seg\ntrain: train/images\nval: val/images\nnc: {len(CLASSES)}\nnames: {CLASSES}\n")
print("train:", n_train, " val:", n_val)

## 4. Synthetic dealt hands
Each one: a real picture with its card, plus 1-3 more real cards cut out (using the outlines), turned up to 60 deg,
and laid partly on top. Covered parts are removed from the cards underneath; a card less than half visible isn't counted (its corners may be hidden).
Training hands only use training cards; the test hands only use held-out cards.

In [ ]:
def load_instances(split_dir):
    out = []
    for p in sorted(glob.glob(f'{split_dir}/images/*.jpg')):
        if os.path.basename(p).startswith('synth_'):
            continue
        img = cv2.imread(p)
        name = os.path.splitext(os.path.basename(p))[0]
        for c, poly in read_seg_label(f'{split_dir}/labels/{name}.txt', img.shape[1], img.shape[0]):
            out.append((p, c, poly))
    return out


def poly_mask(shape, poly):
    m = np.zeros(shape[:2], np.uint8)
    cv2.fillPoly(m, [poly.astype(np.int32)], 1)
    return m.astype(bool)


def mask_to_poly(mask):
    contours = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    c = max(contours, key=cv2.contourArea)
    return cv2.approxPolyDP(c, 1.5, True).reshape(-1, 2).astype(np.float32)


def make_hand(instances, rng, cache):
    # One synthetic dealt hand: a real picture with its card, plus 1-3 more real cards cut out, rotated
    # and laid partly on top, like a dealer spreads cards. Covered parts are cut out of the cards underneath.
    def load(p):
        if p not in cache:
            cache[p] = cv2.imread(p)
        return cache[p]
    base_path, base_c, base_poly = instances[rng.integers(len(instances))]
    img = load(base_path).copy()
    h, w = img.shape[:2]
    cards = [(base_c, poly_mask(img.shape, base_poly))]
    full_area = [cards[0][1].sum()]
    anchor = base_poly.mean(0)
    size = np.sqrt(full_area[0])
    for _ in range(rng.integers(1, 4)):
        p, c, poly = instances[rng.integers(len(instances))]
        src = load(p)
        src_mask = poly_mask(src.shape, poly).astype(np.uint8)
        center = poly.mean(0)
        angle = rng.uniform(-60, 60) + rng.choice([0, 180])
        step = rng.uniform(0.35, 0.8) * size
        direction = rng.uniform(0, 2 * np.pi)
        target = np.clip(anchor + step * np.array([np.cos(direction), np.sin(direction)]), [0.15 * w, 0.15 * h], [0.85 * w, 0.85 * h])
        M = cv2.getRotationMatrix2D(tuple(center), angle, rng.uniform(0.9, 1.1))
        M[:, 2] += target - center
        warped = cv2.warpAffine(src, M, (w, h), flags=cv2.INTER_LINEAR)
        m = cv2.warpAffine(src_mask, M, (w, h), flags=cv2.INTER_NEAREST).astype(bool)
        if m.sum() < 0.3 * src_mask.sum():
            continue                                   # mostly off the picture
        img[m] = warped[m]
        cards = [(cc, cm & ~m) for cc, cm in cards]    # cover what's underneath
        cards.append((c, m)); full_area.append(m.sum())
        anchor = target
    items = [(c, mask_to_poly(m)) for (c, m), a in zip(cards, full_area) if m.sum() >= 0.5 * a]
    return img, items


def make_synthetic(split_dir, out_dir, n, seed):
    rng = np.random.default_rng(seed)
    instances, cache = load_instances(split_dir), {}
    os.makedirs(f'{out_dir}/images', exist_ok=True); os.makedirs(f'{out_dir}/labels', exist_ok=True)
    for i in range(n):
        img, items = make_hand(instances, rng, cache)
        cv2.imwrite(f'{out_dir}/images/synth_{i:04d}.jpg', img)
        write_seg_label(f'{out_dir}/labels/synth_{i:04d}.txt', items, img.shape[1], img.shape[0])


# Training hands use only TRAIN cards; test hands use only held-out VAL cards
make_synthetic('/content/seg/train', '/content/seg/train', n=3 * n_train, seed=0)
shutil.rmtree('/content/hands_test', ignore_errors=True)
make_synthetic('/content/seg/val', '/content/hands_test', n=150, seed=1)
print("Synthetic training hands:", 3 * n_train, "| synthetic test hands:", 150)

plt.figure(figsize=(18, 9))
for i, p in enumerate(sorted(glob.glob('/content/seg/train/images/synth_*.jpg'))[:8]):
    img = cv2.imread(p)
    plt.subplot(2, 4, i + 1)
    draw_items(img, read_seg_label(p.replace('/images/', '/labels/').replace('.jpg', '.txt'), img.shape[1], img.shape[0]),
               os.path.basename(p))
plt.tight_layout(); plt.show()

## 5. Save runs to Google Drive
Keep the Colab tab open while it trains.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/blackjack_runs'
os.makedirs(RUN_DIR, exist_ok=True)
print("Runs will be saved to", RUN_DIR)

## 6. Train (`TRAIN_MINUTES`)
| Trick | Why |
|---|---|
| Segmentation (`-seg`) | Card outlines keep overlapping, turned cards apart |
| `copy_paste`, mode `mixup` | Pastes cards from other pictures on top (not mirrored copies) |
| Synthetic dealt hands | Overlapping cards like a real deal |
| `degrees=45` | Dealt cards are turned |
| Lighting, shift / zoom, mosaic, mixup | Same as before: lighting changes, different poses |
| No flips | A mirrored card isn't a real card |

In [ ]:
model = YOLO(START_WEIGHTS)
model.train(
    data='/content/seg.yaml', imgsz=IMGSZ,
    time=TRAIN_MINUTES / 60, epochs=300, patience=0,     # time wins: epochs get fit to the time
    batch=32, cache='ram', workers=4, seed=0,
    cos_lr=True, warmup_epochs=3,
    # lighting
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,
    # geometry: dealt cards are turned a lot
    degrees=45, translate=0.2, scale=0.5, shear=5, perspective=0.0005,
    fliplr=0.0, flipud=0.0,                              # a mirrored card isn't a real card
    # mixing pictures
    mosaic=1.0, close_mosaic=5, mixup=0.1,
    copy_paste=0.5, copy_paste_mode='mixup',             # paste cards from OTHER pictures (not mirrored copies)
    project=RUN_DIR, name='seg', exist_ok=True)
BEST = f'{RUN_DIR}/seg/weights/best.pt'

### Resume (only if Colab disconnected mid-training)
Rerun the cells above (not training), set `RESUME = True`, run this.

In [ ]:
RESUME = False   # set True only if Colab disconnected during training

if RESUME:
    BEST = f'{RUN_DIR}/seg/weights/best.pt'
    YOLO(f'{RUN_DIR}/seg/weights/last.pt').train(resume=True)

## 7. Check accuracy
- **Single cards:** held-out real pictures (same test as before)
- **Dealt hands:** the 150 synthetic hands made from held-out cards. A hand counts only if every card is read right.

In [ ]:
model = YOLO(BEST)
model.val(data='/content/seg.yaml', imgsz=IMGSZ)

# 1) Single cards (held-out real pictures): most confident card has the right rank
right, total = 0, 0
for img in sorted(glob.glob('/content/seg/val/images/*.jpg')):
    if os.path.basename(img).startswith('synth_'):
        continue
    true = CLASSES[int(open(img.replace('/images/', '/labels/').replace('.jpg', '.txt')).read().split()[0])]
    r = model(img, imgsz=IMGSZ, verbose=False)[0]
    pred = CLASSES[int(r.boxes.cls[r.boxes.conf.argmax()])] if len(r.boxes) else 'none'
    right += pred == true; total += 1
print(f"Single cards: {right}/{total} = {100 * right / max(total, 1):.1f}%")

# 2) Dealt hands (synthetic, from held-out cards): the whole hand read exactly right
exact, cards_right, cards_total, misses = 0, 0, 0, []
for img in sorted(glob.glob('/content/hands_test/images/*.jpg')):
    lab = img.replace('/images/', '/labels/').replace('.jpg', '.txt')
    true = sorted(CLASSES[int(l.split()[0])] for l in open(lab).read().splitlines())
    r = model(img, imgsz=IMGSZ, conf=0.5, verbose=False)[0]
    pred = sorted(CLASSES[int(c)] for c in r.boxes.cls)
    exact += pred == true
    cards_total += len(true)
    left = list(pred)
    for t in true:
        if t in left:
            left.remove(t); cards_right += 1
    if pred != true:
        misses.append((os.path.basename(img), true, pred))
print(f"Dealt hands read exactly right: {exact}/150 = {100 * exact / 150:.1f}%   |   cards found right: {100 * cards_right / cards_total:.1f}%")
for m in misses[:10]:
    print("  ", m)

## See some dealt-hand predictions

In [ ]:
imgs = sorted(glob.glob('/content/hands_test/images/*.jpg'))[:6]
plt.figure(figsize=(18, 9))
for i, img in enumerate(imgs):
    r = model(img, imgsz=IMGSZ, conf=0.5, verbose=False)[0]
    plt.subplot(2, 3, i + 1)
    plt.imshow(r.plot()[:, :, ::-1]); plt.axis('off')
plt.tight_layout(); plt.show()

## 8. Export for the Jetson
ONNX, opset 12. Downloads `best.pt` and `best.onnx`; put them in `HackRPI26/models/` on the Jetson
(replacing the old ones). `card_reader` and `blackjack_game` read the outlines automatically.

In [ ]:
onnx_path = YOLO(BEST).export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
print("Exported:", onnx_path)
shutil.copy(onnx_path, RUN_DIR)   # backup copy in Drive
files.download(BEST)
files.download(onnx_path)